# ASAP8 Detection of Change analysis — dF/F

DoC analysis using the extracted ASAP8 **dF/F signal itself** as the primary response variable. Peak latency/raw-trace panels use the un-baselined mean dF/F. Scalar image/sequence/change/omission response magnitudes use trial-local baseline-subtracted ΔdF/F to isolate the event-evoked component without converting the optical signal to spikes.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, re
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import ndimage
from matplotlib.lines import Line2D
from IPython.display import display, HTML

from vip_slap2_analysis.utils.utils import save_figure
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.voltage.dataset import DEPTH_GROUP_ORDER, build_voltage_session_table, build_voltage_roi_table
from vip_slap2_analysis.voltage.responses import load_response_package, get_mean_response

sns.set_style("white")
plt.rcParams.update({"legend.fontsize":"x-large","axes.labelsize":"xx-large","axes.titlesize":"xx-large","xtick.labelsize":"xx-large","ytick.labelsize":"xx-large"})
display(HTML("<style>.container { width:100% !important; }</style>"))

# 1. Setup

In [ ]:
BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")
SAVE_PATH = Path(r"C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Lab_Meetings\2026-07-28_OPhys_LabMeetingV\figures\voltage_plots")
TARGET_MICE = [852835, 863774]
PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check","volume_imaging"]
TRACE_VARIANT = "dff_robust_f0_trial"
REGISTRATION_FILENAME = "roi_identity_registration.csv"
EXCLUDE_INVALID_ROIS = True
EXPECTED_F0_SMOOTH_SEC = 60.0

SESSION_ORDER = ["A0","A1","A2","B0","B1","B2"]
IMAGE_WINDOW_S = (0.0,0.25)
IMAGE_BASELINE_S = (-0.25,0.0)
IMAGE_CYCLE_S = 0.75
MIN_TRIALS_PER_IMAGE = 5

PEAK_WINDOW_S = (-0.25,0.50)
PEAK_SMOOTH_MS = 10.0
PEAK_MODE = "max"       # "max" = positive maximum; "absolute" = largest |dF/F|
LATENCY_PLOT_WINDOW_S = (-0.25,0.50)

MAX_SEQUENCE_PRESENTATIONS = 12
MIN_EPOCHS_PER_POSITION = 5
MIN_SEQUENCE_POSITIONS = 4
REQUIRE_COMPLETE_SEQUENCE_BLOCK = True
PREFERRED_SLOPE_MODE = "max"  # "max" or "max_abs"

DEPTH_COLORS = {"<100 µm":"#EBA287","100–150 µm":"#d1e2b0",">150 µm":"#7bbcd5"}

def depth_group_from_um(depth):
    depth=float(depth)
    return "<100 µm" if depth < 100 else ("100–150 µm" if depth <= 150 else ">150 µm")

## Canonical DoC tables

In [ ]:
registry = VIPSessionRegistry.from_basepath(BASE_PATH)
sessions = build_voltage_session_table(
    registry,subject_ids=TARGET_MICE,paradigms=PARADIGMS,
    exclude_session_types=EXCLUDE_SESSION_TYPES,trace_variant=TRACE_VARIANT,
    expected_f0_smooth_sec=EXPECTED_F0_SMOOTH_SEC
)
rois = build_voltage_roi_table(
    sessions,registration_filename=REGISTRATION_FILENAME,
    exclude_invalid_rois=EXCLUDE_INVALID_ROIS
)
rois["depth_um"] = pd.to_numeric(rois["depth_um"],errors="coerce")
rois["depth_group"] = rois["depth_um"].map(depth_group_from_um)

print(f"{len(sessions)} sessions · {rois['included'].sum()} included ROI observations")
display(sessions[["subject_id","session_id","session_label","session_order","dmd1_depth_um","dmd2_depth_um"]])
display(rois.loc[rois["included"],["subject_id","session_label","dmd","roi","depth_um","depth_group"]].drop_duplicates().sort_values(["depth_um","subject_id","session_label"]))

In [ ]:
def finish_axis(ax):
    sns.despine(ax=ax); ax.tick_params(axis="both",labelsize=11)
    for spine in ax.spines.values(): spine.set_linewidth(2)

def add_depth_legend(ax,loc="best"):
    ax.legend(handles=[Line2D([0],[0],color=DEPTH_COLORS[g],lw=3,marker="o",mec="black",mew=.6,label=g) for g in DEPTH_GROUP_ORDER],title="Depth",frameon=False,fontsize=9,loc=loc)

def smooth_finite(y,sigma):
    y=np.asarray(y,float).reshape(-1); good=np.isfinite(y)
    if good.sum()<3: return np.full_like(y,np.nan)
    filled=np.interp(np.arange(y.size),np.flatnonzero(good),y[good])
    return ndimage.gaussian_filter1d(filled,max(0,float(sigma)),mode="nearest")

def decode_strings(values):
    return np.asarray([x.decode() if isinstance(x,(bytes,np.bytes_)) else str(x) for x in np.asarray(values).reshape(-1)])

def roi_number(x):
    if isinstance(x,str):
        hits=re.findall(r"\d+",x)
        if not hits: raise ValueError(f"Could not parse ROI label {x!r}")
        return int(hits[-1])
    return int(x)

def select_roi_example(spec):
    roi=roi_number(spec["roi"])
    q=rois[rois["included"].astype(bool) & rois["subject_id"].astype(str).eq(str(spec["mouse"])) & rois["session_label"].astype(str).eq(str(spec["day"])) & rois["dmd"].astype(int).eq(int(spec["dmd"])) & rois["roi"].astype(int).eq(roi)]
    if q.empty:
        avail=rois[rois["included"].astype(bool) & rois["subject_id"].astype(str).eq(str(spec["mouse"])) & rois["session_label"].astype(str).eq(str(spec["day"]))][["dmd","roi"]].drop_duplicates().values.tolist()
        raise ValueError(f"No included ROI matches {spec}. Available DMD/ROI pairs: {avail}")
    r=q.iloc[0]; s=sessions[sessions["session_id"].astype(str).eq(str(r["session_id"]))].iloc[0]
    return r,s

def source_roi_axis(package,dmd,source_roi):
    ids=decode_strings(package[f"DMD{int(dmd)}"]["roi_ids"]); label=f"DMD{int(dmd)}_ROI{int(source_roi)}"
    hits=np.flatnonzero(ids==label)
    if not len(hits):
        parsed=np.array([int(re.findall(r"\d+",x)[-1]) for x in ids],int); hits=np.flatnonzero(parsed==int(source_roi))
    if not len(hits): raise KeyError(f"{label} is absent from kept ROI axis")
    return int(hits[0])

def h5_roi_axis(group,dmd,source_roi):
    ids=decode_strings(group["roi_ids"][:]); label=f"DMD{int(dmd)}_ROI{int(source_roi)}"; hits=np.flatnonzero(ids==label)
    if not len(hits):
        parsed=np.array([int(re.findall(r"\d+",x)[-1]) for x in ids],int); hits=np.flatnonzero(parsed==int(source_roi))
    if not len(hits): raise KeyError(f"{label} is absent from H5 kept ROI axis")
    return int(hits[0])

def reconcile_timebase(t,n):
    t=np.asarray(t,float).reshape(-1)
    if len(t)==n: return t
    if len(t)<2: raise ValueError(f"Cannot reconcile {len(t)} time samples to {n} trace samples")
    dt=float(np.nanmedian(np.diff(t))); zero=min(int(np.nanargmin(np.abs(t))),n-1)
    return (np.arange(n,dtype=float)-zero)*dt

def window_slice(t,window):
    t=np.asarray(t,float); a,b=map(float,window)
    i0=int(np.searchsorted(t,a,side="left")); i1=int(np.searchsorted(t,b,side="left"))
    if i1<=i0: raise ValueError(f"Window {window} has no samples in timebase [{t[0]}, {t[-1]}]")
    return slice(i0,i1)

def window_mean(y,t,window,axis=-1):
    return np.nanmean(np.asarray(y,float)[...,window_slice(t,window)],axis=axis)

def interp_trace(t,y,grid):
    t=np.asarray(t,float); y=np.asarray(y,float); good=np.isfinite(t)&np.isfinite(y)
    return np.interp(grid,t[good],y[good],left=np.nan,right=np.nan) if good.sum()>=2 else np.full_like(grid,np.nan,dtype=float)

def plot_longitudinal(ax,df,metric,ylabel,ylim=None,ax_fontsize=None,error="sem"):
    labels=[s for s in SESSION_ORDER if s in set(df["session_label"].astype(str))]
    xm={s:i for i,s in enumerate(labels)}; xx=np.arange(len(labels))

    tracked=df[
        df["manually_registered"].astype(bool)
        & df["global_cell_id"].astype(str).ne("")
        & df["global_cell_id"].astype(str).ne("nan")
    ].copy()
    tracked=tracked.groupby(
        ["subject_id","global_cell_id"],
        group_keys=False,observed=True
    ).filter(lambda z:z["session_id"].nunique()>=2)

    for (_,cell),g in tracked.groupby(["subject_id","global_cell_id"],observed=True):
        g=g.assign(x=g["session_label"].map(xm)).dropna(subset=["x"]).sort_values("x")
#         if len(g)>1:
#             ax.plot(g["x"],g[metric],color=DEPTH_COLORS[str(g["depth_group"].iloc[0])],
#                     lw=.8,alpha=.12,zorder=1)

    for group,g in df.groupby("depth_group",observed=True):
        if error=="sem":
            q=(g.groupby("session_label")[metric]
               .agg(mean="mean",sem="sem")
               .reindex(labels))
            valid=q["mean"].notna().to_numpy()
            if not valid.any(): continue
            xq=xx[valid]; center=q["mean"].to_numpy()[valid]
            err=q["sem"].fillna(0).to_numpy()[valid]
            lo,hi=center-err,center+err
        elif error=="iqr":
            q=(g.groupby("session_label")[metric]
               .agg(median="median",
                    q25=lambda z:np.nanquantile(z,.25),
                    q75=lambda z:np.nanquantile(z,.75))
               .reindex(labels))
            valid=q["median"].notna().to_numpy()
            if not valid.any(): continue
            xq=xx[valid]; center=q["median"].to_numpy()[valid]
            lo=q["q25"].to_numpy()[valid]; hi=q["q75"].to_numpy()[valid]
        else:
            raise ValueError("error must be 'sem' or 'iqr'")

        c=DEPTH_COLORS[str(group)]
        ax.fill_between(xq,lo,hi,color=c,alpha=.12,lw=0,zorder=2)
        ax.plot(xq,center,"-o",color=c,ms=7,mec="black",mew=.8,lw=3,zorder=3,label=str(group))

    if "A2" in xm and "B0" in xm:
        ax.axvline((xm["A2"]+xm["B0"])/2,color=".65",lw=1,ls=":")

    ax.set(xticks=xx,xticklabels=labels,xlabel="Session",ylabel=ylabel)

    if ax_fontsize is not None:
        ax.set_ylabel(ylabel,fontsize=ax_fontsize)
        ax.set_xlabel("Session",fontsize=ax_fontsize)

    if ylim is not None:
        ax.set_ylim(ylim)

    finish_axis(ax)

# 2. Image responses
## Peak latency from un-baselined mean dF/F

In [ ]:
selected=sessions[sessions["session_label"].astype(str).isin(SESSION_ORDER)]
available_labels=[s for s in SESSION_ORDER if s in set(selected["session_label"].astype(str))]
peak_rows=[]; trace_rows=[]

for session in selected.itertuples(index=False):
    sid=str(session.session_id); pkg=load_response_package(session.mean_npz)
    for r in rois[(rois["session_id"].astype(str)==sid) & rois["included"].astype(bool)].itertuples(index=False):
        dmd,roi,key=int(r.dmd),int(r.roi),f"DMD{int(r.dmd)}"
        if key not in pkg: continue
        image_traces=[]; image_times=[]
        for image in pkg[key].get("image_identity",{}):
            try: t,y=get_mean_response(pkg,dmd=dmd,source_roi=roi,event_type="image",image_name=image)
            except (KeyError,IndexError,ValueError): continue
            t=np.asarray(t,float).reshape(-1); y=np.asarray(y,float).squeeze()
            if y.ndim!=1 or len(y)!=len(t) or len(y)<3: continue
            dt=np.nanmedian(np.diff(t))
            if not np.isfinite(dt) or dt<=0: continue
            ys=smooth_finite(y,(PEAK_SMOOTH_MS/1000)/dt)
            keep=(t>=PEAK_WINDOW_S[0])&(t<=PEAK_WINDOW_S[1])&np.isfinite(ys)
            if not keep.any(): continue
            idx=np.flatnonzero(keep)
            local=np.nanargmax(np.abs(ys[keep])) if PEAK_MODE=="absolute" else np.nanargmax(ys[keep])
            p=idx[local]
            peak_rows.append(dict(subject_id=str(r.subject_id),session_id=sid,session_label=str(r.session_label),session_order=int(r.session_order),dmd=dmd,roi=roi,cell_id=str(r.cell_id),global_cell_id=str(getattr(r,"global_cell_id","")),manually_registered=bool(getattr(r,"manually_registered",False)),depth_um=float(r.depth_um),depth_group=str(r.depth_group),image_name=str(image),peak_latency_s=float(t[p]),peak_dff=float(ys[p])))
            image_traces.append(y); image_times.append(t)
        if image_traces:
            grid=image_times[0]
            stacked=np.vstack([interp_trace(tt,yy,grid) for tt,yy in zip(image_times,image_traces)])
            trace_rows.append(dict(subject_id=str(r.subject_id),session_id=sid,session_label=str(r.session_label),session_order=int(r.session_order),dmd=dmd,roi=roi,cell_id=str(r.cell_id),global_cell_id=str(getattr(r,"global_cell_id","")),manually_registered=bool(getattr(r,"manually_registered",False)),depth_um=float(r.depth_um),depth_group=str(r.depth_group),n_images=len(image_traces),time=grid,mean_image_dff=np.nanmean(stacked,axis=0)))

image_peak_latency=pd.DataFrame(peak_rows); mean_image_trace_df=pd.DataFrame(trace_rows)
if image_peak_latency.empty: raise RuntimeError("No image-response peaks extracted.")

keys=["subject_id","session_id","session_label","session_order","cell_id","global_cell_id","manually_registered"]
cell_session_peak=image_peak_latency.groupby(keys,observed=True,dropna=False).agg(peak_latency_s=("peak_latency_s","median"),peak_latency_q25_s=("peak_latency_s",lambda x:np.nanquantile(x,.25)),peak_latency_q75_s=("peak_latency_s",lambda x:np.nanquantile(x,.75)),median_peak_dff=("peak_dff","median"),n_images=("image_name","nunique"),depth_um=("depth_um","median")).reset_index()
cell_session_peak["depth_group"]=cell_session_peak["depth_um"].map(depth_group_from_um)
tracked_peak=cell_session_peak[cell_session_peak["manually_registered"].astype(bool) & cell_session_peak["global_cell_id"].astype(str).ne("") & cell_session_peak["global_cell_id"].astype(str).ne("nan")].groupby(["subject_id","global_cell_id"],group_keys=False,observed=True).filter(lambda x:x["session_id"].nunique()>=2)

xmap={s:i for i,s in enumerate(available_labels)}; x=np.arange(len(available_labels))
fig,ax=plt.subplots(figsize=(5.2,3.6))
for (_,cell),g in tracked_peak.groupby(["subject_id","global_cell_id"],observed=True):
    g=g.assign(x=g["session_label"].map(xmap)).dropna(subset=["x"]).sort_values("x")
    if len(g)>1: ax.plot(g["x"],1000*g["peak_latency_s"],color=DEPTH_COLORS[str(g["depth_group"].iloc[0])],lw=.8,alpha=.16,zorder=1)
for group,g in tracked_peak.groupby("depth_group",observed=True):
    q=g.groupby("session_label")["peak_latency_s"].agg(median="median",q25=lambda z:np.nanquantile(z,.25),q75=lambda z:np.nanquantile(z,.75)).reindex(available_labels)
    valid=q["median"].notna().to_numpy(); xx=x[valid]; c=DEPTH_COLORS[str(group)]
    ax.fill_between(xx,1000*q["q25"].to_numpy()[valid],1000*q["q75"].to_numpy()[valid],color=c,alpha=.4,lw=0)
    ax.plot(xx,1000*q["median"].to_numpy()[valid],"-o",color=c,lw=3,ms=7,mec="black",mew=.8,label=str(group),zorder=3)
ax.axhline(0,color=".5",lw=1,ls="--"); ax.axhline(250,color="k",lw=.5,dashes=[6,3],zorder=0)
if "A2" in xmap and "B0" in xmap: ax.axvline((xmap["A2"]+xmap["B0"])/2,color=".65",lw=1,ls=":")
ax.set(xticks=x,xticklabels=available_labels,xlabel="Session",ylabel="Time of maximum mean dF/F\nrelative to image onset (ms)",title="Image-response timing across sessions")
finish_axis(ax); ax.legend(title="Depth",frameon=False,fontsize=9); fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,"latency_shift"),formats=[".pdf"],dpi=300); plt.show()

## Mean image dF/F traces for every neuron across days

In [ ]:
grid=np.linspace(LATENCY_PLOT_WINDOW_S[0],LATENCY_PLOT_WINDOW_S[1],751)
fig,axs=plt.subplots(2,3,figsize=(6.0,5.0),sharex=True,sharey=True); axs=axs.ravel()

for ax,label in zip(axs,SESSION_ORDER):
    q=mean_image_trace_df[mean_image_trace_df["session_label"].astype(str).eq(label)]
    for row in q.itertuples(index=False):
        ax.plot(grid,interp_trace(row.time,row.mean_image_dff,grid),color=DEPTH_COLORS[str(row.depth_group)],lw=1,alpha=.5)
    for group,g in q.groupby("depth_group",observed=True):
        a=np.vstack([interp_trace(row.time,row.mean_image_dff,grid) for row in g.itertuples(index=False)])
        ax.plot(grid,np.nanmedian(a,axis=0),color=DEPTH_COLORS[str(group)],lw=3,label=str(group),zorder=3)
    ax.axvspan(0,IMAGE_WINDOW_S[1],color=".75",alpha=.12,lw=0); ax.axvline(0,color=".45",lw=1,ls="--")
    ax.set_title(label); finish_axis(ax)
for ax in axs[3:]: ax.set_xlabel("Time from image onset (s)")
axs[0].set_ylabel("Mean \u0394F/F$_{0}$"); axs[3].set_ylabel("Mean \u0394F/F$_{0}$"); add_depth_legend(axs[2])
fig.suptitle("Mean image response",y=0.975,fontsize=20)
fig.tight_layout(); save_figure(fig,os.path.join(SAVE_PATH,"mean_image_dff_all_neurons"),formats=[".pdf"],dpi=300); plt.show()
filen = 'MeanImageResponse'
save_figure(fig,os.path.join(SAVE_PATH,filen),formats=['.pdf'],dpi=300)

## Within-neuron change in the mean image dF/F trace

In [ ]:
pairs=[("A0","A2"),("B0","B2")]
fig,axs=plt.subplots(2,1,figsize=(4.5,5.0),sharex=True,sharey=True)

for ax,(first,last) in zip(axs,pairs):
    diffs=[]
    tracked=mean_image_trace_df[mean_image_trace_df["manually_registered"].astype(bool) & mean_image_trace_df["global_cell_id"].astype(str).ne("") & mean_image_trace_df["global_cell_id"].astype(str).ne("nan")]
    for (mouse,cell),g in tracked.groupby(["subject_id","global_cell_id"],observed=True):
        a=g[g["session_label"].astype(str).eq(first)]; b=g[g["session_label"].astype(str).eq(last)]
        if a.empty or b.empty: continue
        a,b=a.iloc[0],b.iloc[0]; y0=interp_trace(a["time"],a["mean_image_dff"],grid); y1=interp_trace(b["time"],b["mean_image_dff"],grid)
        depth=float(np.nanmedian([a["depth_um"],b["depth_um"]])); group=depth_group_from_um(depth); diff=y1-y0
        diffs.append((group,diff)); ax.plot(grid,diff,color=DEPTH_COLORS[group],lw=.75,alpha=.18)
    for group in DEPTH_GROUP_ORDER:
        arr=[d for g,d in diffs if g==group]
        if arr: ax.plot(grid,np.nanmedian(np.vstack(arr),axis=0),color=DEPTH_COLORS[group],lw=2.8,label=group)
    ax.axhline(0,color=".5",lw=1,ls="--"); ax.axvline(0,color=".45",lw=1,ls="--"); ax.axvspan(0,IMAGE_WINDOW_S[1],color=".75",alpha=.10,lw=0)
    ax.set(xlabel="Time from image onset (s)",title=f"{last} − {first}"); finish_axis(ax)
axs[0].set_ylabel("\u0394 mean \u0394F/F$_{0}$",labelpad=0); add_depth_legend(axs[1])
axs[1].set_ylabel("\u0394 mean \u0394F/F$_{0}$",labelpad=0)
# fig.suptitle("Within-neuron change in mean image response across days",y=0.95,fontsize=18)
fig.tight_layout(); save_figure(fig,os.path.join(SAVE_PATH,"mean_image_dff_longitudinal_difference"),formats=[".pdf",'.png'],dpi=300); plt.show()

## Trial-wise image variability in ΔdF/F

For every ordinary image trial, response magnitude is `mean dF/F[0–250 ms] − mean dF/F[−250–0 ms]`. FVE and RMS are calculated directly from these trial-wise optical response values.

In [ ]:
metric_keys=["subject_id","session_id","session_label","session_order","dmd","roi","cell_id","global_cell_id","manually_registered","depth_um","depth_group"]
trial_rows=[]; metric_rows=[]

for session in sessions[sessions["session_label"].astype(str).isin(SESSION_ORDER)].itertuples(index=False):
    sid=str(session.session_id); session_rois=rois[rois["included"].astype(bool) & rois["session_id"].astype(str).eq(sid)]
    with h5py.File(session.single_trial_h5,"r") as h5:
        stored_t=np.asarray(h5["timebase_sec/image"][:],float)
        for r in session_rois.itertuples(index=False):
            group=h5[f"DMD{int(r.dmd)}"]; axis=h5_roi_axis(group,int(r.dmd),int(r.roi))
            ys=[]; labels=[]
            for key in group["image_identity"]:
                sub=group["image_identity"][key]; n=int(sub["traces"].shape[0])
                if n<MIN_TRIALS_PER_IMAGE: continue
                t=reconcile_timebase(stored_t,int(sub["traces"].shape[-1]))
                base=np.nanmean(np.asarray(sub["traces"][:,axis,window_slice(t,IMAGE_BASELINE_S)],float),axis=1)
                resp=np.nanmean(np.asarray(sub["traces"][:,axis,window_slice(t,IMAGE_WINDOW_S)],float),axis=1)
                image=str(sub.attrs.get("image_name",key)); delta=resp-base
                ys.append(delta); labels.extend([image]*len(delta))
            if not ys: continue
            y=np.concatenate(ys); labels=np.asarray(labels); good=np.isfinite(y); y=y[good]; labels=labels[good]
            if len(y)<2 or len(np.unique(labels))<2: continue
            grand=float(np.mean(y)); total=float(np.mean((y-grand)**2))
            stats=pd.DataFrame({"image":labels,"y":y}).groupby("image")["y"].agg(["mean","size"])
            between=float(np.sum(stats["size"]*(stats["mean"]-grand)**2)/len(y))
            meta=dict(subject_id=str(r.subject_id),session_id=sid,session_label=str(r.session_label),session_order=int(r.session_order),dmd=int(r.dmd),roi=int(r.roi),cell_id=str(r.cell_id),global_cell_id=str(getattr(r,"global_cell_id","")),manually_registered=bool(getattr(r,"manually_registered",False)),depth_um=float(r.depth_um),depth_group=str(r.depth_group))
            metric_rows.append({**meta,"mean_image_delta_dff":grand,"total_dff_variance":total,"identity_dff_variance":between,"image_rms_dff":float(np.sqrt(between)),"image_fve":float(between/total) if total>0 else np.nan,"n_trials":len(y),"n_images":len(stats)})
            trial_rows.append(pd.DataFrame({**{k:[v]*len(y) for k,v in meta.items()},"image_label":labels,"image_delta_dff":y}))

image_trial_df=pd.concat(trial_rows,ignore_index=True); image_metrics=pd.DataFrame(metric_rows)
registered_metrics=image_metrics[image_metrics["manually_registered"].astype(bool) & image_metrics["global_cell_id"].astype(str).ne("") & image_metrics["global_cell_id"].astype(str).ne("nan")].copy()
cell_metrics=registered_metrics.groupby(["subject_id","global_cell_id","depth_group"],observed=True).agg(depth_um=("depth_um","median"),total_dff_variance=("total_dff_variance","median"),image_fve=("image_fve","median"),image_rms_dff=("image_rms_dff","median"),n_sessions=("session_id","nunique")).reset_index()
display(image_metrics.head())

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(9.2,3.6))
order=[g for g in DEPTH_GROUP_ORDER if g in set(cell_metrics["depth_group"].astype(str))]
sns.violinplot(data=cell_metrics,x="depth_group",y="total_dff_variance",order=order,palette=DEPTH_COLORS,inner=None,width=.5,linewidth=1.5,ax=axs[0])
for coll in axs[0].collections: coll.set_alpha(.25); coll.set_edgecolor("black")
rng=np.random.default_rng(8); xpos=np.array([order.index(str(x)) for x in cell_metrics["depth_group"]],float)+rng.uniform(-.12,.12,len(cell_metrics))
axs[0].scatter(xpos,cell_metrics["total_dff_variance"],s=42,c=[DEPTH_COLORS[str(x)] for x in cell_metrics["depth_group"]],edgecolor="black",linewidth=.6,zorder=3)
axs[0].set(xlabel="Depth bin",ylabel="Median trial-wise \u0394F/F$_{0}$ variance\nacross sessions",title="Total image-response variability"); finish_axis(axs[0])
plot_longitudinal(axs[1],image_metrics,"total_dff_variance","Trial-wise \u0394F/F$_{0}$ variance"); axs[1].set_title("Image-response variability\nacross sessions"); add_depth_legend(axs[1])
fig.tight_layout(); save_figure(fig,os.path.join(SAVE_PATH,"image_trial_dff_variance"),formats=[".pdf"],dpi=300); plt.show()

fig,axs=plt.subplots(1,2,figsize=(9.2,3.6))
plot_longitudinal(axs[0],image_metrics,"image_fve","FVE")
plot_longitudinal(axs[1],image_metrics,"image_rms_dff","RMS mod. (\u0394F/F$_{0}$)")
axs[0].axhline(0,color=".6",lw=1,ls="--"); axs[1].axhline(0,color=".6",lw=1,ls="--")
axs[0].set_title("FVE by image identity"); axs[1].set_title("Image-identity response magnitude"); add_depth_legend(axs[1])
fig.tight_layout(); save_figure(fig,os.path.join(SAVE_PATH,"image_identity_dff_fve_rms"),formats=[".pdf"],dpi=300); plt.show()

# 3. Image-sequence dynamics

Sequence responses now come directly from the extracted sequence dF/F summaries. Each position is `mean dF/F[0–250 ms] − mean dF/F[−250–0 ms]`. Image-specific slopes are weighted by the number of contributing epochs; the population panel uses one fixed-effect slope per neuron/session after removing each image's intercept.

In [ ]:
seq_pos_rows=[]; seq_slope_rows=[]

for session in sessions[sessions["session_label"].astype(str).isin(SESSION_ORDER)].itertuples(index=False):
    sid=str(session.session_id); pkg=load_response_package(session.sequence_npz); t0=np.asarray(pkg["timebase_sec"]["image"],float)
    for r in rois[rois["included"].astype(bool) & rois["session_id"].astype(str).eq(sid)].itertuples(index=False):
        dmd=int(r.dmd); roi=int(r.roi); key=f"DMD{dmd}"
        if key not in pkg: continue
        axis=source_roi_axis(pkg,dmd,roi)
        meta=dict(subject_id=str(r.subject_id),session_id=sid,session_label=str(r.session_label),session_order=int(r.session_order),dmd=dmd,roi=roi,cell_id=str(r.cell_id),global_cell_id=str(getattr(r,"global_cell_id","")),manually_registered=bool(getattr(r,"manually_registered",False)),depth_um=float(r.depth_um),depth_group=str(r.depth_group))
        for image,entry in pkg[key].get("image_identity",{}).items():
            rep=entry.get("repeated",{})
            if not rep or "mean" not in rep: continue
            traces=np.asarray(rep["mean"],float)[:,axis,:]; pos=np.asarray(rep["positions"],int); counts=np.asarray(rep["counts"],int)
            t=reconcile_timebase(t0,traces.shape[-1]); delta=window_mean(traces,t,IMAGE_WINDOW_S)-window_mean(traces,t,IMAGE_BASELINE_S)
            keep=(pos<=MAX_SEQUENCE_PRESENTATIONS)&(counts>=MIN_EPOCHS_PER_POSITION)&np.isfinite(delta)
            p,y,w=pos[keep].astype(float),delta[keep].astype(float),counts[keep].astype(float)
            for pp,yy,ww in zip(p,y,w): seq_pos_rows.append({**meta,"sequence_image":str(image),"sequence_position":int(pp),"mean_delta_dff":float(yy),"n_epochs":int(ww)})
            if len(p)<MIN_SEQUENCE_POSITIONS or len(np.unique(p))<MIN_SEQUENCE_POSITIONS: continue
            slope,intercept=np.polyfit(p,y,1,w=np.sqrt(w)); pred=intercept+slope*p; ss_res=np.sum(w*(y-pred)**2); ybar=np.average(y,weights=w); ss_tot=np.sum(w*(y-ybar)**2)
            seq_slope_rows.append({**meta,"sequence_image":str(image),"sequence_slope_dff_per_presentation":float(slope),"sequence_intercept_dff":float(intercept),"sequence_r2":float(1-ss_res/ss_tot) if ss_tot>0 else np.nan,"n_positions":len(p),"min_epochs_per_position":int(w.min())})

sequence_position_df=pd.DataFrame(seq_pos_rows); sequence_slopes=pd.DataFrame(seq_slope_rows)

# One pooled slope per neuron/session, with image identity treated as a fixed intercept.
neuron_rows=[]
group_keys=["subject_id","session_id","session_label","session_order","dmd","roi","cell_id","global_cell_id","manually_registered","depth_um","depth_group"]
for keys,g in sequence_position_df.groupby(group_keys,observed=True,dropna=False):
    if g["sequence_image"].nunique()<2: continue
    pieces=[]
    for image,h in g.groupby("sequence_image",observed=True):
        x=h["sequence_position"].to_numpy(float); y=h["mean_delta_dff"].to_numpy(float); w=h["n_epochs"].to_numpy(float)
        if len(np.unique(x))<2: continue
        xc=x-np.average(x,weights=w); yc=y-np.average(y,weights=w)
        pieces.append((xc,yc,w))
    if not pieces: continue
    xc=np.concatenate([p[0] for p in pieces]); yc=np.concatenate([p[1] for p in pieces]); w=np.concatenate([p[2] for p in pieces])
    denom=np.sum(w*xc**2)
    if denom<=0: continue
    row=dict(zip(group_keys,keys)); row["sequence_slope_dff_per_presentation"]=float(np.sum(w*xc*yc)/denom); row["n_images"]=g["sequence_image"].nunique(); row["n_position_means"]=len(g)
    neuron_rows.append(row)

neuron_sequence_slopes=pd.DataFrame(neuron_rows)
print(f"{len(sequence_slopes):,} image-specific slopes · {len(neuron_sequence_slopes):,} neuron/session pooled slopes")
display(neuron_sequence_slopes.head())

## Selectable sequence example

In [ ]:
SEQUENCE_EXAMPLE = dict(mouse=863774, day="A2", dmd=1, roi=0, image=None)

r, s = select_roi_example(SEQUENCE_EXAMPLE)
sid = str(r["session_id"])
dmd = int(r["dmd"])
roi = int(r["roi"])

q = sequence_slopes[
    sequence_slopes["session_id"].astype(str).eq(sid)
    & sequence_slopes["dmd"].eq(dmd)
    & sequence_slopes["roi"].eq(roi)
].copy()

if q.empty:
    raise ValueError(f"No sequence slopes available for {SEQUENCE_EXAMPLE}")

if SEQUENCE_EXAMPLE["image"] is None:
    ex = q.loc[q["sequence_slope_dff_per_presentation"].abs().idxmax()]
else:
    stem = Path(str(SEQUENCE_EXAMPLE["image"]).replace("\\", "/")).stem
    hit = q[
        q["sequence_image"].astype(str)
        .map(lambda x: Path(x.replace("\\", "/")).stem)
        .eq(stem)
    ]
    if hit.empty:
        raise ValueError(
            f"Image {SEQUENCE_EXAMPLE['image']!r} not found. "
            f"Available: {q['sequence_image'].unique().tolist()}"
        )
    ex = hit.iloc[0]

# ---------- Load selected sequence ----------
pkg = load_response_package(s["sequence_npz"])
dpkg = pkg[f"DMD{dmd}"]
stem = Path(str(ex["sequence_image"]).replace("\\", "/")).stem

ikey = next(
    (
        k for k in dpkg.get("image_identity", {})
        if Path(str(k).replace("\\", "/")).stem == stem
    ),
    None,
)
if ikey is None:
    raise KeyError(f"{stem!r} not found in {s['sequence_npz']}")

rep = dpkg["image_identity"][ikey]["repeated"]
axis = source_roi_axis(pkg, dmd, roi)

traces = np.asarray(rep["mean"], float)[:, axis, :]
std_traces = (
    np.asarray(rep["std"], float)[:, axis, :]
    if "std" in rep else None
)
n_finite = (
    np.asarray(rep["n_finite"], float)[:, axis, :]
    if "n_finite" in rep else None
)

pos = np.asarray(rep["positions"], int)
counts = np.asarray(rep["counts"], int)

t = reconcile_timebase(
    pkg["timebase_sec"]["image"],
    traces.shape[-1],
)

response = window_mean(traces, t, IMAGE_WINDOW_S)
baseline = window_mean(traces, t, IMAGE_BASELINE_S)
delta = response - baseline

keep = (
    (pos <= MAX_SEQUENCE_PRESENTATIONS)
    & (counts >= MIN_EPOCHS_PER_POSITION)
    & np.isfinite(delta)
)

pos = pos[keep]
counts = counts[keep]
traces = traces[keep]
delta = delta[keep]

if std_traces is not None:
    std_traces = std_traces[keep]
if n_finite is not None:
    n_finite = n_finite[keep]

order = np.argsort(pos)

pos = pos[order]
counts = counts[order]
traces = traces[order]
delta = delta[order]

if std_traces is not None:
    std_traces = std_traces[order]
if n_finite is not None:
    n_finite = n_finite[order]

c = DEPTH_COLORS[str(r["depth_group"])]

# ---------- SEM for ΔdF/F ----------
delta_sem = None

if std_traces is not None:
    if n_finite is None:
        n_finite = np.broadcast_to(
            counts[:, None],
            std_traces.shape,
        ).astype(float)

    point_sem = std_traces / np.sqrt(
        np.maximum(n_finite, 1)
    )

    resp_sem = np.nanmean(
        point_sem[:, window_slice(t, IMAGE_WINDOW_S)],
        axis=1,
    )
    base_sem = np.nanmean(
        point_sem[:, window_slice(t, IMAGE_BASELINE_S)],
        axis=1,
    )

    delta_sem = np.sqrt(
        resp_sem**2 + base_sem**2
    )

# ============================================================
# 1. Sequence-position response + linear adaptation fit
# ============================================================
fig, ax = plt.subplots(figsize=(4.7, 3.6))

ax.errorbar(
    pos,
    delta,
    yerr=delta_sem,
    fmt="-o",
    color=c,
    lw=2.2,
    ms=6,
    mec="black",
    mew=0.7,
    ecolor=c,
    elinewidth=1,
    capsize=2.5,
    zorder=3,
)

xx = np.linspace(
    pos.min(),
    pos.max(),
    200,
)

ax.plot(
    xx,
    ex["sequence_intercept_dff"]
    + ex["sequence_slope_dff_per_presentation"] * xx,
    color="black",
    lw=2,
    ls="--",
    label="Linear fit",
    zorder=4,
)

ax.set(
    xlabel="Presentation after image change",
    ylabel="Image response (ΔdF/F)",
    title=(
        f'Mouse {r["subject_id"]} · {r["session_label"]} · '
        f'DMD{dmd} ROI{roi}\n'
        f'{stem} · slope='
        f'{ex["sequence_slope_dff_per_presentation"]:.4f} '
        f'ΔdF/F/presentation'
    ),
)

ax.set_xticks(pos)

finish_axis(ax)
ax.legend(frameon=False, fontsize=8)

fig.tight_layout()

save_figure(
    fig,
    os.path.join(
        SAVE_PATH,
        "sequence_example_dff_response",
    ),
    formats=[".pdf"],
    dpi=300,
)

plt.show()


# ============================================================
# 2. Contiguous raw dF/F sequence
#
# Keep the FRONT of each aligned trace intact.
# Only truncate the BACK at +IMAGE_CYCLE_S.
# Then concatenate chunks end-to-end and recompute image-onset
# positions in the resulting time axis.
# ============================================================

dt = float(np.nanmedian(np.diff(t)))

# Back-end truncation only.
chunk_mask = t < IMAGE_CYCLE_S
chunk_t = t[chunk_mask]

concat_y = []
image_onsets = []

cursor = 0.0

for y in traces:

    y_chunk = y[chunk_mask]

    # Image onset is wherever t=0 falls within this chunk.
    onset_offset = -chunk_t[0]

    image_onsets.append(
        cursor + onset_offset
    )

    concat_y.append(y_chunk)

    # Advance cursor by exactly the duration of this truncated trace.
    cursor += len(y_chunk) * dt

concat_y = np.concatenate(concat_y)

# Continuous plotting timebase.
concat_t = np.arange(
    len(concat_y),
    dtype=float,
) * dt

# Put the first sample at zero for a simple contiguous axis.
image_onsets = np.asarray(image_onsets)

fig, ax = plt.subplots(
    figsize=(9.0, 3.4)
)

ax.plot(
    concat_t,
    concat_y,
    color="black",
    lw=1.1,
)

# Image windows at their NEW positions in the concatenated trace.
for onset in image_onsets:

    ax.axvspan(
        onset,
        onset + IMAGE_WINDOW_S[1],
        color=c,
        alpha=0.10,
        lw=0,
        zorder=0,
    )

    ax.axvline(
        onset,
        color="0.75",
        lw=0.5,
        zorder=0,
    )

ax.set(
    xlabel="Time through concatenated sequence (s)",
    ylabel="Mean dF/F",
    title=(
        f'Sequence dF/F · Mouse {r["subject_id"]} '
        f'{r["session_label"]} · DMD{dmd} ROI{roi} · {stem}'
    ),
)

finish_axis(ax)
fig.tight_layout()

save_figure(
    fig,
    os.path.join(
        SAVE_PATH,
        "sequence_example_raw_dff",
    ),
    formats=[".pdf"],
    dpi=300,
)

plt.show()

In [ ]:
# Compare facilitating vs adapting sequence examples on common axes

SEQUENCE_COMPARE = {
    "Adapting": dict(mouse=852835, day="A2", dmd=1, roi=2, image=None),
    "Facilitating":     dict(mouse=852835, day="A1", dmd=2, roi=0, image=None),
}

COMPARE_COLORS = {
    "Adapting": "#4C9BD5",
    "Facilitating": "#E6866A",
}


def load_sequence_comparison(spec, mode):
    r, s = select_roi_example(spec)

    sid = str(r["session_id"])
    dmd = int(r["dmd"])
    roi = int(r["roi"])

    q = sequence_slopes[
        sequence_slopes["session_id"].astype(str).eq(sid)
        & sequence_slopes["dmd"].eq(dmd)
        & sequence_slopes["roi"].eq(roi)
    ].copy()

    if q.empty:
        raise ValueError(f"No sequence slopes available for {spec}")

    if spec["image"] is None:
        if mode == "Facilitating":
            ex = q.loc[q["sequence_slope_dff_per_presentation"].idxmax()]
        else:
            ex = q.loc[q["sequence_slope_dff_per_presentation"].idxmin()]
    else:
        stem = Path(str(spec["image"]).replace("\\", "/")).stem
        hit = q[
            q["sequence_image"].astype(str)
            .map(lambda x: Path(x.replace("\\", "/")).stem)
            .eq(stem)
        ]

        if hit.empty:
            raise ValueError(
                f"Image {spec['image']!r} not found. "
                f"Available: {q['sequence_image'].unique().tolist()}"
            )

        ex = hit.iloc[0]

    # ---------- Sequence data ----------
    pkg = load_response_package(s["sequence_npz"])
    dpkg = pkg[f"DMD{dmd}"]

    stem = Path(
        str(ex["sequence_image"]).replace("\\", "/")
    ).stem

    ikey = next(
        (
            k for k in dpkg.get("image_identity", {})
            if Path(str(k).replace("\\", "/")).stem == stem
        ),
        None,
    )

    if ikey is None:
        raise KeyError(
            f"{stem!r} not found in {s['sequence_npz']}"
        )

    rep = dpkg["image_identity"][ikey]["repeated"]
    axis = source_roi_axis(pkg, dmd, roi)

    traces = np.asarray(
        rep["mean"],
        float,
    )[:, axis, :]

    std_traces = (
        np.asarray(rep["std"], float)[:, axis, :]
        if "std" in rep
        else None
    )

    n_finite = (
        np.asarray(rep["n_finite"], float)[:, axis, :]
        if "n_finite" in rep
        else None
    )

    pos = np.asarray(
        rep["positions"],
        int,
    )

    counts = np.asarray(
        rep["counts"],
        int,
    )

    t = reconcile_timebase(
        pkg["timebase_sec"]["image"],
        traces.shape[-1],
    )

    # ---------- Response magnitude ----------
    response = window_mean(
        traces,
        t,
        IMAGE_WINDOW_S,
    )

    baseline = window_mean(
        traces,
        t,
        IMAGE_BASELINE_S,
    )

    delta = response - baseline

    keep = (
        (pos <= MAX_SEQUENCE_PRESENTATIONS)
        & (counts >= MIN_EPOCHS_PER_POSITION)
        & np.isfinite(delta)
    )

    pos = pos[keep]
    counts = counts[keep]
    traces = traces[keep]
    delta = delta[keep]

    if std_traces is not None:
        std_traces = std_traces[keep]

    if n_finite is not None:
        n_finite = n_finite[keep]

    order = np.argsort(pos)

    pos = pos[order]
    counts = counts[order]
    traces = traces[order]
    delta = delta[order]

    if std_traces is not None:
        std_traces = std_traces[order]

    if n_finite is not None:
        n_finite = n_finite[order]

    # ---------- Approximate ΔdF/F SEM ----------
    delta_sem = None

    if std_traces is not None:

        if n_finite is None:
            n_finite = np.broadcast_to(
                counts[:, None],
                std_traces.shape,
            ).astype(float)

        point_sem = (
            std_traces
            / np.sqrt(np.maximum(n_finite, 1))
        )

        response_sem = np.nanmean(
            point_sem[
                :,
                window_slice(
                    t,
                    IMAGE_WINDOW_S,
                )
            ],
            axis=1,
        )

        baseline_sem = np.nanmean(
            point_sem[
                :,
                window_slice(
                    t,
                    IMAGE_BASELINE_S,
                )
            ],
            axis=1,
        )

        delta_sem = np.sqrt(
            response_sem**2
            + baseline_sem**2
        )

    # ---------- Contiguous raw sequence ----------
    # Keep the entire FRONT of each event-aligned trace.
    # Truncate only the back end at +750 ms.
    dt = float(
        np.nanmedian(np.diff(t))
    )

    chunk_mask = t < IMAGE_CYCLE_S
    chunk_t = t[chunk_mask]

    concat_y = []
    image_onsets = []

    cursor = 0.0

    for y in traces:

        y_chunk = y[chunk_mask]

        # Position of t=0 inside this retained chunk.
        onset_offset = -chunk_t[0]

        image_onsets.append(
            cursor + onset_offset
        )

        concat_y.append(
            y_chunk
        )

        cursor += len(y_chunk) * dt

    concat_y = np.concatenate(
        concat_y
    )

    concat_t = (
        np.arange(
            len(concat_y),
            dtype=float,
        )
        * dt
    )

    return {
        "label": mode,
        "r": r,
        "s": s,
        "ex": ex,
        "stem": stem,
        "pos": pos,
        "delta": delta,
        "delta_sem": delta_sem,
        "concat_t": concat_t,
        "concat_y": concat_y,
        "image_onsets": np.asarray(image_onsets),
    }


examples = {
    label: load_sequence_comparison(
        spec,
        label,
    )
    for label, spec in SEQUENCE_COMPARE.items()
}


# ============================================================
# 1. Facilitating vs adapting response slopes
# ============================================================

fig, ax = plt.subplots(
    figsize=(5.5, 3.75)
)

for label, d in examples.items():

    color = COMPARE_COLORS[label]

    ax.errorbar(
        d["pos"],
        d["delta"],
        yerr=d["delta_sem"],
        fmt="-o",
        color=color,
        lw=2.2,
        ms=6,
        mec="black",
        mew=0.7,
        ecolor=color,
        elinewidth=1,
        capsize=2.5,
        label=(
            f'{label} '
            f'({d["ex"]["sequence_slope_dff_per_presentation"]:.4f})'
        ),
        zorder=3,
    )

    xx = np.linspace(
        d["pos"].min(),
        d["pos"].max(),
        200,
    )

    ax.plot(
        xx,
        d["ex"]["sequence_intercept_dff"]
        + d["ex"]["sequence_slope_dff_per_presentation"] * xx,
        color=color,
        lw=1.8,
        ls="--",
        alpha=0.9,
        zorder=2,
    )


ax.axhline(
    0,
    color="0.6",
    lw=1,
    ls=":",
)

ax.set(
    xlabel="Presentation after image change",
    ylabel="Image response (ΔdF/F)",
    title="Facilitating vs adapting sequence responses",
)

finish_axis(ax)

ax.legend(
    frameon=False,
    fontsize=8,
)

fig.tight_layout()

save_figure(
    fig,
    os.path.join(
        SAVE_PATH,
        "sequence_facilitating_vs_adapting_response",
    ),
    formats=[".pdf"],
    dpi=300,
)

plt.show()


# ============================================================
# 2. Facilitating vs adapting contiguous raw dF/F sequences
# ============================================================

fig, ax = plt.subplots(
    figsize=(5.5, 3.75)
)

# Draw image epochs once using the first example.
reference = next(
    iter(examples.values())
)

for onset in reference["image_onsets"]:

    ax.axvspan(
        onset,
        onset + IMAGE_WINDOW_S[1],
        color="0.75",
        alpha=0.12,
        lw=0,
        zorder=0,
    )

    ax.axvline(
        onset,
        color="0.82",
        lw=0.5,
        zorder=0,
    )


for label, d in examples.items():

    ax.plot(
        d["concat_t"],
        d["concat_y"],
        color=COMPARE_COLORS[label],
        lw=1.25,
        alpha=0.95,
        label=(
            f'{label}: '
            f'Mouse {d["r"]["subject_id"]} '
            f'{d["r"]["session_label"]} · '
            f'DMD{int(d["r"]["dmd"])} '
            f'ROI{int(d["r"]["roi"])} · '
            f'{d["stem"]}'
        ),
    )


ax.set(
    xlabel="Time through concatenated sequence (s)",
    ylabel="Mean dF/F",
    title="Facilitating vs adapting sequence dF/F",
)

finish_axis(ax)

ax.legend(
    frameon=False,
    fontsize=7,
)

fig.tight_layout()

save_figure(
    fig,
    os.path.join(
        SAVE_PATH,
        "sequence_facilitating_vs_adapting_raw_dff",
    ),
    formats=[".pdf"],
    dpi=300,
)

plt.show()

## Sequence slope by session and depth

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 3.8))

xmap = {s: i for i, s in enumerate(SESSION_ORDER)}
x = np.arange(len(SESSION_ORDER))

for group, g in neuron_sequence_slopes.groupby("depth_group", observed=True):
    c = DEPTH_COLORS[str(group)]

    q = (
        g.groupby("session_label")["sequence_slope_dff_per_presentation"]
        .agg(
            median="median",
            q25=lambda z: np.nanquantile(z, 0.25),
            q75=lambda z: np.nanquantile(z, 0.75),
        )
        .reindex(SESSION_ORDER)
    )

    valid = q["median"].notna().to_numpy()
    xx = x[valid]

    ax.fill_between(
        xx,
        q["q25"].to_numpy()[valid],
        q["q75"].to_numpy()[valid],
        color=c,
        alpha=0.12,
        lw=0,
        zorder=1,
    )

    ax.plot(
        xx,
        q["median"].to_numpy()[valid],
        "-o",
        color=c,
        lw=2.8,
        ms=6,
        mec="black",
        mew=0.7,
        label=str(group),
        zorder=2,
    )

ax.axhline(
    0,
    color=".55",
    lw=1,
    ls="--",
)

ax.axvline(
    2.5,
    color=".7",
    lw=1,
    ls=":",
)

ax.set(
    xticks=x,
    xticklabels=SESSION_ORDER,
    xlabel="Session",
    ylabel="Sequence slope\n(\u0394F/F$_{0}$ / presentation)",
    title="Image-sequence slope across sessions",
)

finish_axis(ax)
add_depth_legend(ax)

fig.tight_layout()

save_figure(
    fig,
    os.path.join(
        SAVE_PATH,
        "sequence_dff_slopes_by_depth",
    ),
    formats=[".pdf",'.png'],
    dpi=300,
)

plt.show()

## Longitudinal change in each neuron's strongest image-sequence slope

In [ ]:
def preferred_sequence_block(df,labels):
    start=labels[0]; q=df[df["session_label"].astype(str).isin(labels)&df["manually_registered"].astype(bool)&df["global_cell_id"].astype(str).ne("")&df["global_cell_id"].astype(str).ne("nan")].copy()
    s=q[q["session_label"].astype(str).eq(start)].copy(); s["rank_value"]=s["sequence_slope_dff_per_presentation"].abs() if PREFERRED_SLOPE_MODE=="max_abs" else s["sequence_slope_dff_per_presentation"]
    pref=s.sort_values("rank_value").drop_duplicates(["subject_id","global_cell_id"],keep="last")[["subject_id","global_cell_id","sequence_image","sequence_slope_dff_per_presentation","depth_group"]].rename(columns={"sequence_image":"preferred_image","sequence_slope_dff_per_presentation":"start_slope","depth_group":"start_depth_group"})
    q=q.merge(pref,on=["subject_id","global_cell_id"],how="inner"); q=q[q["sequence_image"].eq(q["preferred_image"])]
    if REQUIRE_COMPLETE_SEQUENCE_BLOCK:
        complete=q.groupby(["subject_id","global_cell_id"],observed=True)["session_label"].nunique(); keep=complete[complete==len(labels)].index
        q=q.set_index(["subject_id","global_cell_id"]).loc[keep].reset_index() if len(keep) else q.iloc[0:0]
    q["delta_sequence_slope"]=q["sequence_slope_dff_per_presentation"]-q["start_slope"]; q["block"]=f"{labels[0]}→{labels[-1]}"
    return q

preferred_sequence_change=pd.concat([preferred_sequence_block(sequence_slopes,["A0","A1","A2"]),preferred_sequence_block(sequence_slopes,["B0","B1","B2"])],ignore_index=True)
blocks=[("A0→A2",["A0","A1","A2"]),("B0→B2",["B0","B1","B2"])]
fig,axs=plt.subplots(1,2,figsize=(8.4,3.5),sharey=True)
for ax,(block,labels) in zip(axs,blocks):
    q=preferred_sequence_change[preferred_sequence_change["block"].eq(block)].copy(); xm={s:i for i,s in enumerate(labels)}
    for (_,cell),g in q.groupby(["subject_id","global_cell_id"],observed=True):
        g=g.assign(x=g["session_label"].map(xm)).dropna(subset=["x"]).sort_values("x"); color=DEPTH_COLORS[str(g["start_depth_group"].iloc[0])]
        ax.plot(g["x"],g["delta_sequence_slope"],"-o",color=color,lw=1,alpha=.22,ms=4)
    for group,g in q.groupby("start_depth_group",observed=True):
        s=g.groupby("session_label")["delta_sequence_slope"].agg(median="median",q25=lambda z:np.nanquantile(z,.25),q75=lambda z:np.nanquantile(z,.75)).reindex(labels)
        valid=s["median"].notna().to_numpy(); xx=np.arange(len(labels))[valid]; color=DEPTH_COLORS[str(group)]
        ax.fill_between(xx,s["q25"].to_numpy()[valid],s["q75"].to_numpy()[valid],color=color,alpha=.12,lw=0)
        ax.plot(xx,s["median"].to_numpy()[valid],"-o",color=color,lw=3,ms=7,mec="black",mew=.8,label=str(group),zorder=3)
    ax.axhline(0,color=".55",lw=1,ls="--"); ax.set(xticks=np.arange(len(labels)),xticklabels=labels,xlabel="Session",title=block); finish_axis(ax)
axs[0].set_ylabel("Δ preferred-image sequence slope\nfrom first day (ΔdF/F / presentation)"); add_depth_legend(axs[1])
fig.tight_layout(); save_figure(fig,os.path.join(SAVE_PATH,"preferred_sequence_dff_slope_change"),formats=[".pdf"],dpi=300); plt.show()

# 4. Image-change responses

For each change-aligned dF/F trial, the **pre-change image response** is the −750 to −500 ms image minus its immediately preceding gray baseline (−1000 to −750 ms). The **change response** is 0–250 ms minus the immediately preceding gray baseline (−250–0 ms). The primary contrast is change response − pre-change response.

In [ ]:
CHANGE_EXAMPLE=dict(mouse=863774,day="B0",dmd=1,roi=2)
change_rows=[]; change_curve_rows=[]

for session in sessions[sessions["session_label"].astype(str).isin(SESSION_ORDER)].itertuples(index=False):
    sid=str(session.session_id); session_rois=rois[rois["included"].astype(bool)&rois["session_id"].astype(str).eq(sid)]
    with h5py.File(session.single_trial_h5,"r") as h5:
        stored_t=np.asarray(h5["timebase_sec/change"][:],float)
        for r in session_rois.itertuples(index=False):
            group=h5[f"DMD{int(r.dmd)}"]; axis=h5_roi_axis(group,int(r.dmd),int(r.roi)); sub=group["change"]
            traces=np.asarray(sub["traces"][:,axis,:],float); t=reconcile_timebase(stored_t,traces.shape[-1])
            pre=window_mean(traces,t,(-.75,-.50))#-window_mean(traces,t,(-1.0,-.75))
            change=window_mean(traces,t,(0,.25))#-window_mean(traces,t,(-.25,0))
            base=window_mean(traces,t,(-.25,0)); centered=traces#-base[:,None]; 
            mean_curve=np.nanmean(centered,axis=0)
            meta=dict(subject_id=str(r.subject_id),session_id=sid,session_label=str(r.session_label),session_order=int(r.session_order),dmd=int(r.dmd),roi=int(r.roi),cell_id=str(r.cell_id),global_cell_id=str(getattr(r,"global_cell_id","")),manually_registered=bool(getattr(r,"manually_registered",False)),depth_um=float(r.depth_um),depth_group=str(r.depth_group))
            change_rows.append({**meta,"pre_change_dff":float(np.nanmean(pre)),"change_dff":float(np.nanmean(change)),"change_minus_pre_dff":float(np.nanmean(change-pre)),"n_changes":len(traces)})
            change_curve_rows.append({**meta,"time":t,"mean_centered_dff":mean_curve,"n_changes":len(traces)})

change_metrics=pd.DataFrame(change_rows); change_curve_df=pd.DataFrame(change_curve_rows)
display(change_metrics.head())

## Selectable raw mean dF/F change example

In [ ]:
r,s=select_roi_example(CHANGE_EXAMPLE)
with h5py.File(s["single_trial_h5"],"r") as h5:
    group=h5[f"DMD{int(r['dmd'])}"]; axis=h5_roi_axis(group,int(r["dmd"]),int(r["roi"])); sub=group["change"]
    traces=np.asarray(sub["traces"][:,axis,:],float); t=reconcile_timebase(h5["timebase_sec/change"][:],traces.shape[-1])
y=np.nanmean(traces,axis=0); sem=np.nanstd(traces,axis=0,ddof=1)/np.sqrt(np.maximum(1,np.sum(np.isfinite(traces),axis=0))); c=DEPTH_COLORS[str(r["depth_group"])]
fig,ax=plt.subplots(figsize=(6.4,3.6)); ax.fill_between(t,y-sem,y+sem,color=".55",alpha=.18,lw=0); ax.plot(t,y,color="black",lw=1.35)
ax.axvspan(-.75,-.50,color=".75",alpha=.20,lw=0); ax.axvspan(0,.25,color=c,alpha=.18,lw=0); ax.axvline(0,color=".35",lw=1,ls="--")
ax.set(xlabel="Time from image change (s)",ylabel="Mean dF/F",title=f'Change dF/F · Mouse {r["subject_id"]} {r["session_label"]} · DMD{int(r["dmd"])} ROI{int(r["roi"])} · {r["depth_group"]}\nn={len(traces)} changes')
finish_axis(ax); fig.tight_layout(); save_figure(fig,os.path.join(SAVE_PATH,"change_example_raw_dff"),formats=[".pdf"],dpi=300); plt.show()

## Pre-change versus change ΔdF/F

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(8.4,3.6))
for group,g in change_metrics.groupby("depth_group",observed=True):
    axs[0].scatter(g["pre_change_dff"],g["change_dff"],s=28,color=DEPTH_COLORS[str(group)],alpha=.45,edgecolor="none")
vals=np.r_[change_metrics["pre_change_dff"],change_metrics["change_dff"]]; lo=np.nanpercentile(vals,1); hi=np.nanpercentile(vals,99); pad=.05*(hi-lo if hi>lo else 1); lim=(lo-pad,hi+pad)
axs[0].plot(lim,lim,color=".5",ls="--",lw=1); axs[0].set(xlim=lim,ylim=lim,xlabel="Pre-change response (ΔdF/F)",ylabel="Change response (ΔdF/F)",title="Neuron/session responses"); finish_axis(axs[0])
plot_longitudinal(axs[1],change_metrics,"change_minus_pre_dff","Change − pre-change (ΔdF/F)"); axs[1].axhline(0,color=".55",lw=1,ls="--"); axs[1].set_title("Change-specific response"); add_depth_legend(axs[1])
fig.tight_layout(); save_figure(fig,os.path.join(SAVE_PATH,"change_dff_pre_vs_change"),formats=[".pdf"],dpi=300); plt.show()

## Change-aligned dF/F response shape across neurons

In [ ]:
plot_grid=np.linspace(-1.0,.75,1751)
fig,axs=plt.subplots(2,3,figsize=(11.5,6.0),sharex=True,sharey=True); axs=axs.ravel()
for ax,label in zip(axs,SESSION_ORDER):
    q=change_curve_df[change_curve_df["session_label"].astype(str).eq(label)]
    for group,g in q.groupby("depth_group",observed=True):
        a=np.vstack([interp_trace(row.time,row.mean_centered_dff,plot_grid) for row in g.itertuples(index=False)])
        med=np.nanmedian(a,axis=0); lo=np.nanquantile(a,.25,axis=0); hi=np.nanquantile(a,.75,axis=0); c=DEPTH_COLORS[str(group)]
        ax.fill_between(plot_grid,lo,hi,color=c,alpha=.15,lw=0); ax.plot(plot_grid,med,color=c,lw=2.2,label=str(group))
    ax.axvspan(-.75,-.50,color=".78",alpha=.16,lw=0); ax.axvspan(0,.25,color=".78",alpha=.25,lw=0); ax.axvline(0,color=".35",lw=1,ls="--"); ax.set_title(label); finish_axis(ax)
for ax in axs[3:]: ax.set_xlabel("Time from change (s)")
axs[0].set_ylabel("\u0394F/F$_{0}$"); axs[3].set_ylabel("\u0394F/F$_{0}$"); add_depth_legend(axs[2])
fig.suptitle("Change-aligned \u0394F/F$_{0}$ profiles by session and depth",y=0.95,fontsize=18); fig.tight_layout(); save_figure(fig,os.path.join(SAVE_PATH,"change_dff_session_depth"),formats=[".pdf",'.png'],dpi=300); plt.show()

# 5. Omission responses

The **pre-omission image** is quantified as its 250-ms image response above its own gray baseline. The **omission response** is mean dF/F over the full 0–750 ms omitted cycle relative to −250–0 ms. Ramping is late (375–750 ms) − early (0–375 ms). The **post-omission image response** is 750–1000 ms relative to the immediately preceding 500–750 ms omission activity.

In [ ]:
OMISSION_EXAMPLE=dict(mouse=852835,day="B2",dmd=1,roi=1)
omission_rows=[]; omission_curve_rows=[]

for session in sessions[sessions["session_label"].astype(str).isin(SESSION_ORDER)].itertuples(index=False):
    sid=str(session.session_id); session_rois=rois[rois["included"].astype(bool)&rois["session_id"].astype(str).eq(sid)]
    with h5py.File(session.single_trial_h5,"r") as h5:
        stored_t=np.asarray(h5["timebase_sec/omission"][:],float)
        for r in session_rois.itertuples(index=False):
            group=h5[f"DMD{int(r.dmd)}"]; axis=h5_roi_axis(group,int(r.dmd),int(r.roi)); sub=group["omission"]
            traces=np.asarray(sub["traces"][:,axis,:],float); t=reconcile_timebase(stored_t,traces.shape[-1])
            pre=window_mean(traces,t,(-.75,-.50))#-window_mean(traces,t,(-1.0,-.75))
            om_base=window_mean(traces,t,(-.25,0)); omission=window_mean(traces,t,(0,.75))#-om_base
            early=window_mean(traces,t,(0,.375))#-om_base;
            late=window_mean(traces,t,(.375,.75))#-om_base
            post=window_mean(traces,t,(.75,1.0))#-window_mean(traces,t,(.50,.75))
            centered=traces#-om_base[:,None]; 
            mean_curve=np.nanmean(centered,axis=0)
            meta=dict(subject_id=str(r.subject_id),session_id=sid,session_label=str(r.session_label),session_order=int(r.session_order),dmd=int(r.dmd),roi=int(r.roi),cell_id=str(r.cell_id),global_cell_id=str(getattr(r,"global_cell_id","")),manually_registered=bool(getattr(r,"manually_registered",False)),depth_um=float(r.depth_um),depth_group=str(r.depth_group))
            omission_rows.append({**meta,"pre_omission_dff":float(np.nanmean(pre)),"omission_dff":float(np.nanmean(omission)),"omission_minus_pre_dff":float(np.nanmean(omission-pre)),"omission_ramp_dff":float(np.nanmean(late-early)),"post_omission_dff":float(np.nanmean(post)),"post_minus_pre_dff":float(np.nanmean(post-pre)),"n_omissions":len(traces)})
            omission_curve_rows.append({**meta,"time":t,"mean_centered_dff":mean_curve,"n_omissions":len(traces)})

omission_metrics=pd.DataFrame(omission_rows); omission_curve_df=pd.DataFrame(omission_curve_rows)
display(omission_metrics.head())

## Selectable raw mean dF/F omission example

In [ ]:
r,s=select_roi_example(OMISSION_EXAMPLE)
with h5py.File(s["single_trial_h5"],"r") as h5:
    group=h5[f"DMD{int(r['dmd'])}"]; axis=h5_roi_axis(group,int(r["dmd"]),int(r["roi"])); sub=group["omission"]
    traces=np.asarray(sub["traces"][:,axis,:],float); t=reconcile_timebase(h5["timebase_sec/omission"][:],traces.shape[-1])
y=np.nanmean(traces,axis=0); sem=np.nanstd(traces,axis=0,ddof=1)/np.sqrt(np.maximum(1,np.sum(np.isfinite(traces),axis=0))); c=DEPTH_COLORS[str(r["depth_group"])]
fig,ax=plt.subplots(figsize=(6.8,3.6)); ax.fill_between(t,y-sem,y+sem,color=".55",alpha=.18,lw=0); ax.plot(t,y,color="black",lw=1.35)
ax.axvspan(-.75,-.50,color=".75",alpha=.20,lw=0); ax.axvspan(0,.75,color=".85",alpha=.14,lw=0); ax.axvspan(.75,1.0,color=c,alpha=.18,lw=0); ax.axvline(0,color=".35",lw=1,ls="--"); ax.axvline(.75,color=".45",lw=1,ls=":")
ax.set(xlabel="Time from expected omitted-image onset (s)",ylabel="Mean dF/F",title=f'Omission dF/F · Mouse {r["subject_id"]} {r["session_label"]} · DMD{int(r["dmd"])} ROI{int(r["roi"])} · {r["depth_group"]}\nn={len(traces)} omissions')
finish_axis(ax); fig.tight_layout(); save_figure(fig,os.path.join(SAVE_PATH,"omission_example_raw_dff"),formats=[".pdf"],dpi=300); plt.show()
fig.tight_layout()

## Pre-omission, omission, ramping, and post-omission ΔdF/F

In [ ]:
# ================== OMISSION RESPONSE SCATTERS ==================

# Omission vs pre-omission image
fig,ax=plt.subplots(figsize=(4.2,3.6))
for group,g in omission_metrics.groupby("depth_group",observed=True):
    c=DEPTH_COLORS[str(group)]
    ax.scatter(g["pre_omission_dff"],g["omission_dff"],s=28,color=c,alpha=.45,edgecolor="none")

vals=np.r_[omission_metrics["pre_omission_dff"],omission_metrics["omission_dff"]]
lo,hi=np.nanpercentile(vals,[1,99]); pad=.05*(hi-lo if hi>lo else 1); lim=(lo-pad,hi+pad)
ax.plot(lim,lim,color=".5",ls="--",lw=1)
ax.set(xlim=lim,ylim=lim,xlabel="Pre-omission image response ΔF/F$_0$",
       ylabel="Omission response ΔF/F$_0$")
finish_axis(ax); add_depth_legend(ax)
fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,"omission_vs_pre_scatter"),formats=[".pdf"],dpi=300)
plt.show()


# Post-omission image vs pre-omission image
fig,ax=plt.subplots(figsize=(4.2,3.6))
for group,g in omission_metrics.groupby("depth_group",observed=True):
    c=DEPTH_COLORS[str(group)]
    ax.scatter(g["pre_omission_dff"],g["post_omission_dff"],s=28,color=c,alpha=.45,edgecolor="none")

vals=np.r_[omission_metrics["pre_omission_dff"],omission_metrics["post_omission_dff"]]
lo,hi=np.nanpercentile(vals,[1,99]); pad=.05*(hi-lo if hi>lo else 1); lim=(lo-pad,hi+pad)
ax.plot(lim,lim,color=".5",ls="--",lw=1)
ax.set(xlim=lim,ylim=lim,xlabel="Pre-omission image response ΔF/F$_0$",
       ylabel="Post-omission image response ΔF/F$_0$")
finish_axis(ax); add_depth_legend(ax)
fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,"post_omission_vs_pre_scatter"),formats=[".pdf"],dpi=300)
plt.show()


# ================== LONGITUDINAL OMISSION METRICS ==================

# Omission response
fig,ax=plt.subplots(figsize=(5.2,3.6))
plot_longitudinal(ax,omission_metrics,"omission_minus_pre_dff","Omission − pre ΔF/F$_0$")
ax.axhline(0,color=".55",lw=1,ls="--")
ax.set_title("Omission response")
add_depth_legend(ax)
fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,"omission_response_session_depth"),formats=[".pdf"],dpi=300)
plt.show()


# Within-omission ramp
fig,ax=plt.subplots(figsize=(5.2,3.6))
plot_longitudinal(ax,omission_metrics,"omission_ramp_dff","Late − early omission ΔF/F$_0$")
ax.axhline(0,color=".55",lw=1,ls="--")
ax.set_title("Within-omission ramp")
add_depth_legend(ax)
fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,"omission_ramp_session_depth"),formats=[".pdf"],dpi=300)
plt.show()


# Post-omission image response
fig,ax=plt.subplots(figsize=(5.2,3.6))
plot_longitudinal(ax,omission_metrics,"post_minus_pre_dff","Post-image − pre-image ΔF/F$_0$")
ax.axhline(0,color=".55",lw=1,ls="--")
ax.set_title("Post-omission image response")
add_depth_legend(ax)
fig.tight_layout()
save_figure(fig,os.path.join(SAVE_PATH,"post_omission_response_session_depth"),formats=[".pdf"],dpi=300)
plt.show()

In [ ]:
# ========== POST-OMISSION VS PRE-OMISSION IMAGE RESPONSE ==========

fig,ax=plt.subplots(figsize=(5,3))

plot_longitudinal(
    ax,
    omission_metrics,
    "post_minus_pre_dff",
    "Post − pre (\u0394F/F$_0$)"
)

ax.axhline(0,color=".55",lw=1,ls="--")
ax.set_title("Image reappearance")

add_depth_legend(ax)

fig.tight_layout()
save_figure(
    fig,
    os.path.join(SAVE_PATH,"post_vs_pre_omission_image_response"),
    formats=[".pdf",".png"],
    dpi=300
)
plt.show()

## Omission-aligned dF/F response shape across neurons

In [ ]:
plot_grid=np.linspace(-1.0,1.25,2251)
fig,axs=plt.subplots(2,3,figsize=(11.5,6.0),sharex=True,sharey=True); axs=axs.ravel()
for ax,label in zip(axs,SESSION_ORDER):
    q=omission_curve_df[omission_curve_df["session_label"].astype(str).eq(label)]
    for group,g in q.groupby("depth_group",observed=True):
        a=np.vstack([interp_trace(row.time,row.mean_centered_dff,plot_grid) for row in g.itertuples(index=False)])
        med=np.nanmedian(a,axis=0); lo=np.nanquantile(a,.25,axis=0); hi=np.nanquantile(a,.75,axis=0); c=DEPTH_COLORS[str(group)]
        ax.fill_between(plot_grid,lo,hi,color=c,alpha=.15,lw=0); ax.plot(plot_grid,pd.DataFrame(med).rolling(10,min_periods=1).mean(),color=c,lw=2.2,label=str(group))
    ax.axvspan(-.75,-.50,color=".78",alpha=.16,lw=0); ax.axvspan(0,.75,color=".88",alpha=.12,lw=0); ax.axvspan(.75,1.0,color=".78",alpha=.22,lw=0)
    ax.axvline(0,color=".35",lw=1,ls="--"); ax.axvline(.75,color=".5",lw=1,ls=":"); ax.set_title(label); finish_axis(ax)
for ax in axs[3:]: ax.set_xlabel("Time from omission (s)")
axs[0].set_ylabel("\u0394F/F$_{0}$"); axs[3].set_ylabel("\u0394F/F$_{0}$"); add_depth_legend(axs[2])
fig.suptitle("Omission-aligned \u0394F/F$_{0}$ profiles by session and depth",y=0.95,fontsize=18); fig.tight_layout(); save_figure(fig,os.path.join(SAVE_PATH,"omission_dff_session_depth"),formats=[".pdf",'.png'],dpi=300); plt.show()

## Composite summary figure for image variability slide

Three-panel summary: one example neuron's image-wise mean dF/F responses, aggregate total response variability by depth, and image-identity FVE across sessions.

In [ ]:
SUMMARY_FIG_EXAMPLE = dict(mouse=863774, day="A2", dmd=1, roi=0)
SUMMARY_FIG_SAVE_NAME = "image_variability_summary_figure"

image_colors = [
    "#c5cae9", "#ffcdd2", "#c8e6c9", "#ffe0b2",
    "#e1bee7", "#d7ccc8", "#cfd8dc", "#b2ebf2",
]

# ---------- Example neuron: image-wise mean responses ----------
r, s = select_roi_example(SUMMARY_FIG_EXAMPLE)
pkg = load_response_package(s["mean_npz"])
key = f"DMD{int(r['dmd'])}"

image_curves = []
if key not in pkg:
    raise KeyError(f"{key} not present in {s['mean_npz']}")

for image in pkg[key].get("image_identity", {}):
    try:
        t, y = get_mean_response(
            pkg,
            dmd=int(r["dmd"]),
            source_roi=int(r["roi"]),
            event_type="image",
            image_name=image,
        )
    except (KeyError, IndexError, ValueError):
        continue

    t = np.asarray(t, float).reshape(-1)
    y = np.asarray(y, float).squeeze()

    if y.ndim != 1 or len(y) != len(t):
        continue

    yy = interp_trace(
        t,
        y,
        np.linspace(
            LATENCY_PLOT_WINDOW_S[0],
            LATENCY_PLOT_WINDOW_S[1],
            751,
        ),
    )

    image_curves.append((
        str(image),
        yy,
        np.nanmean(
            yy[
                (np.linspace(
                    LATENCY_PLOT_WINDOW_S[0],
                    LATENCY_PLOT_WINDOW_S[1],
                    751,
                ) >= IMAGE_WINDOW_S[0])
                &
                (np.linspace(
                    LATENCY_PLOT_WINDOW_S[0],
                    LATENCY_PLOT_WINDOW_S[1],
                    751,
                ) < IMAGE_WINDOW_S[1])
            ]
        )
        -
        np.nanmean(
            yy[
                (np.linspace(
                    LATENCY_PLOT_WINDOW_S[0],
                    LATENCY_PLOT_WINDOW_S[1],
                    751,
                ) >= IMAGE_BASELINE_S[0])
                &
                (np.linspace(
                    LATENCY_PLOT_WINDOW_S[0],
                    LATENCY_PLOT_WINDOW_S[1],
                    751,
                ) < IMAGE_BASELINE_S[1])
            ]
        ),
    ))

if not image_curves:
    raise RuntimeError(
        f"No image-wise mean responses found for {SUMMARY_FIG_EXAMPLE}"
    )

example_grid = np.linspace(
    LATENCY_PLOT_WINDOW_S[0],
    LATENCY_PLOT_WINDOW_S[1],
    751,
)

# Keep the order deterministic and assign one of the eight requested colors.
image_curves = sorted(image_curves, key=lambda x: x[0])

# ---------- Figure layout ----------
# Top row: example image-wise responses + total variability.
# Bottom row: FVE across sessions spanning the full width.
fig = plt.figure(figsize=(6.5, 5.5))
gs = fig.add_gridspec(
    2, 2,
    height_ratios=[1.0, 0.95],
    hspace=0.42,
    wspace=0.34,
)

ax_example = fig.add_subplot(gs[0, 0])
ax_var = fig.add_subplot(gs[0, 1])
ax_fve = fig.add_subplot(gs[1, :])

# ---------- Example neuron: image-wise mean responses ----------
for i, (image, y, magnitude) in enumerate(image_curves):
    ax_example.plot(
        example_grid,
        y,
        color=image_colors[i % len(image_colors)],
        lw=1.5,
        alpha=0.95,
        label=Path(image.replace("\\", "/")).stem,
    )

ax_example.axvspan(
    0,
    IMAGE_WINDOW_S[1],
    color="0.75",
    alpha=0.10,
    lw=0,
)
ax_example.axvline(
    0,
    color="0.45",
    lw=1,
    ls="--",
)

ax_example.set_title(
    "Single-neuron image-wise\nmean responses",fontsize=15
)
ax_example.set_xlabel(
    "Time from image onset (s)",fontsize=12
)
ax_example.set_ylabel(
    "Mean \u0394F/F$_{0}$",fontsize=12
)

finish_axis(ax_example)

# ax_example.text(
#     0.02,
#     0.98,
#     (
#         f'Mouse {r["subject_id"]} · {r["session_label"]}\n'
#         f'DMD{int(r["dmd"])} ROI{int(r["roi"])} · {r["depth_group"]}'
#     ),
#     transform=ax_example.transAxes,
#     ha="left",
#     va="top",
#     fontsize=7.5,
#     color="0.25",
# )

# ---------- Aggregate total response variability ----------
order = [
    g for g in DEPTH_GROUP_ORDER
    if g in set(cell_metrics["depth_group"].astype(str))
]

sns.violinplot(
    data=cell_metrics,
    x="depth_group",
    y="total_dff_variance",
    order=order,
    palette=DEPTH_COLORS,
    inner=None,
    width=0.5,
    linewidth=1.2,
    ax=ax_var,
)

for coll in ax_var.collections:
    coll.set_alpha(0.7)
    coll.set_edgecolor("black")

rng = np.random.default_rng(8)
xpos = (
    np.array(
        [order.index(str(x)) for x in cell_metrics["depth_group"]],
        float,
    )
    + rng.uniform(-0.12, 0.12, len(cell_metrics))
)

ax_var.scatter(
    xpos,
    cell_metrics["total_dff_variance"],
    s=28,
    c=[
        DEPTH_COLORS[str(x)]
        for x in cell_metrics["depth_group"]
    ],
    edgecolor="black",
    linewidth=0.45,
    zorder=3,
    alpha=0.85,
)

ax_var.set_title(
    "Total trial-wise\nresponse variability",fontsize=15
)
ax_var.set_xlabel(
    "Depth bin",fontsize=12
)
ax_var.set_ylabel(
    "Variance",fontsize=12
)

finish_axis(ax_var)

# ---------- FVE across sessions ----------
plot_longitudinal(
    ax_fve,
    image_metrics,
    "image_fve",
    "FVE",
    ax_fontsize=12
)

ax_fve.axhline(
    0,
    color="0.55",
    lw=1,
    ls="--",
)

ax_fve.set_title(
    "FVE over sessions",fontsize=15
)
add_depth_legend(
    ax_fve,
    loc="upper right",
)

fig.tight_layout()

save_figure(
    fig,
    os.path.join(
        SAVE_PATH,
        SUMMARY_FIG_SAVE_NAME,
    ),
    formats=[".pdf",'.png'],
    dpi=300,
)

plt.show()